# 🤖 OpenAI Clinical Text Extraction - Interactive Notebook

This notebook demonstrates **cloud-based clinical text extraction** using OpenAI's API with the comprehensive EchoReport Pydantic model. Leverages state-of-the-art language models for high-quality extraction.

## Features
- **State-of-the-Art Models**: Access to GPT-4o, GPT-4o-mini, and other advanced models
- **High Accuracy**: Typically superior extraction quality compared to local models
- **Scalable Processing**: Handle large datasets efficiently with cloud infrastructure
- **Clinical-Grade Schema**: EchoReport model with comprehensive cardiac parameters
- **Cost Optimization**: Choose between quality (GPT-4o) and efficiency (GPT-4o-mini)

## Prerequisites
1. Valid OpenAI API key set in environment variables
2. CSV file with echo report text data
3. ExtraCTOps venv_openai environment activated
4. Sufficient OpenAI API credits

## Use Cases
- High-accuracy extraction for research publications
- Large-scale clinical data processing
- Comparative analysis with local models
- Production deployments requiring maximum quality

## ⚠️ Important Notes
- **API Costs**: Processing incurs OpenAI API charges
- **Data Privacy**: Data is sent to OpenAI's servers (review their privacy policy)
- **Rate Limits**: Respect OpenAI's rate limiting policies

## 1. Setup and Configuration

Import libraries and configure the extraction parameters optimized for OpenAI.

In [ ]:
# Standard library imports
import os
import sys
import asyncio
import json
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Any, Optional

# Data processing
import pandas as pd

# Add project root to Python path
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# ExtraCTOps imports
from utils.ExtraCTOps_loops import ProcessingConfig, ExtraCTOpsProcessor
from the_pydantics.EchoReport import EchoReport

print("✅ All libraries imported successfully!")
print(f"📁 Project root: {project_root}")
print(f"🐍 Python version: {sys.version}")
print(f"📊 Pandas version: {pd.__version__}")

## 2. API Key Validation

Check if OpenAI API key is properly configured and validate access.

In [ ]:
# Check and validate OpenAI API key
def validate_openai_setup():
    """Validate OpenAI API key and show configuration."""
    api_key = os.getenv("OPENAI_API_KEY")
    
    if not api_key:
        print("❌ OPENAI_API_KEY not found!")
        print("\n🔧 To set up your API key:")
        print("1. Get your API key from: https://platform.openai.com/api-keys")
        print("2. Set environment variable:")
        print("   export OPENAI_API_KEY='your-api-key-here'")
        print("3. Or create a .env file in the project root with:")
        print("   OPENAI_API_KEY=your-api-key-here")
        return False
    else:
        print(f"✅ OpenAI API key found!")
        print(f"   Key preview: {api_key[:8]}...{api_key[-4:]}")
        
        # Test API access (optional - comment out to skip)
        try:
            import openai
            client = openai.OpenAI(api_key=api_key)
            
            # Simple test call
            response = client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=[{"role": "user", "content": "Hello"}],
                max_tokens=5
            )
            print("✅ API access verified!")
            return True
            
        except Exception as e:
            print(f"⚠️  API key found but test failed: {e}")
            print("   The key might be invalid or you may have insufficient credits")
            return True  # Continue anyway - might work for actual processing
    
    return False

# Validate setup
api_available = validate_openai_setup()

if not api_available:
    print("\n⏸️ Setup incomplete - please configure your OpenAI API key first")

## 3. Configuration Parameters

Set up extraction configuration optimized for OpenAI cloud processing.

In [ ]:
# Configuration optimized for OpenAI cloud processing
CONFIG = {
    # Data settings
    "TEXT_COLUMN": "echo_report_text",
    "UID_COLUMN": "patient_id",
    
    # Processing settings (can be higher for cloud API)
    "BATCH_SIZE": 5,  # Higher batch size for cloud processing
    "BACKUP_INTERVAL": 10,
    "MAX_RETRIES": 3,  # More retries for network issues
    
    # LLM settings
    "TEMPERATURE": 0.1,  # Low temperature for medical accuracy
    "MAX_TOKENS": 3000,
    "OPENAI_MODEL": "gpt-4o-mini",  # Cost-effective default, change to "gpt-4o" for higher quality
    
    # Medical extraction prompts
    "SYSTEM_MESSAGE": """You are an expert cardiologist and medical data extraction specialist. 
Extract structured echocardiogram information from clinical reports with high accuracy. 
Focus on cardiac anatomy, function, measurements, and pathology. 
Return only valid JSON that matches the provided schema exactly.""",
    
    "PRE_PROMPT": """Analyze this echocardiogram report and extract all relevant cardiac information. 
Include measurements with units, anatomical descriptions, functional assessments, and any abnormalities. 
Be precise with medical terminology and numerical values. Return valid JSON only."""
}

# Setup output directory
output_dir = project_root / "exports" / "openai_extractions"
output_dir.mkdir(parents=True, exist_ok=True)
CONFIG["OUTPUT_DIR"] = output_dir

print("🤖 OPENAI EXTRACTION CONFIGURATION")
print("=" * 40)
for key, value in CONFIG.items():
    if key not in ["SYSTEM_MESSAGE", "PRE_PROMPT"]:  # Skip long text
        print(f"   {key}: {value}")

print(f"\n📁 Output directory: {CONFIG['OUTPUT_DIR']}")
print(f"🫀 Using EchoReport model with {len(EchoReport.model_fields)} main sections")

# Model information and pricing
model_info = {
    "gpt-4o": {"quality": "Highest", "speed": "Medium", "cost": "Higher"},
    "gpt-4o-mini": {"quality": "High", "speed": "Fast", "cost": "Lower"},
    "gpt-4": {"quality": "Very High", "speed": "Slower", "cost": "Highest"},
    "gpt-3.5-turbo": {"quality": "Good", "speed": "Fastest", "cost": "Lowest"}
}

current_model = CONFIG["OPENAI_MODEL"]
if current_model in model_info:
    info = model_info[current_model]
    print(f"\n📊 Selected Model: {current_model}")
    print(f"   Quality: {info['quality']}")
    print(f"   Speed: {info['speed']}")
    print(f"   Cost: {info['cost']}")

## 4. Create or Load Sample Data

Create sample echo report data for testing, or modify to load your own CSV file.

In [ ]:
def create_sample_echo_data():
    """Create realistic sample echo report data for testing."""
    sample_reports = [
        {
            "patient_id": "ECHO_001",
            "study_date": "2024-01-15",
            "echo_report_text": """
ECHOCARDIOGRAM REPORT

Patient: 45-year-old male
Indication: Chest pain, rule out cardiac cause

FINDINGS:
Left Ventricle: The left ventricle is normal in size. Left ventricular systolic function is normal with an estimated ejection fraction of 65%. No regional wall motion abnormalities. LV diastolic volume 110 mL, systolic volume 38 mL.

Right Ventricle: The right ventricle is normal in size and systolic function.

Atria: The left atrium is mildly dilated. Right atrium is normal in size.

Valves: 
- Mitral valve is structurally normal with mild regurgitation
- Tricuspid valve shows mild regurgitation with estimated PA pressure 25 mmHg
- Aortic valve is structurally normal, trileaflet, no stenosis or regurgitation
- Pulmonary valve is normal with trivial regurgitation

Aorta: Aortic root measures 32 mm, ascending aorta 28 mm. Left aortic arch.

No pericardial effusion. No evidence of pulmonary hypertension.

IMPRESSION: Normal left ventricular size and systolic function. Mild left atrial dilation. Mild mitral and tricuspid regurgitation.
            """
        },
        {
            "patient_id": "ECHO_002", 
            "study_date": "2024-01-16",
            "echo_report_text": """
ECHOCARDIOGRAM REPORT

Patient: 62-year-old female
Indication: Hypertension, assessment of cardiac function

FINDINGS:
Left Ventricle: Moderate left ventricular hypertrophy. Estimated ejection fraction 45%, mildly depressed systolic function. LV diastolic volume 145 mL, systolic volume 80 mL.

Right Ventricle: Normal right ventricular size and function.

Atria: Both atria are moderately dilated. Left atrial volume indexed 38 mL/m².

Valves:
- Mitral valve shows mild stenosis and moderate regurgitation
- Aortic valve has mild stenosis with peak gradient 35 mmHg, mean gradient 20 mmHg
- Tricuspid regurgitation is moderate with elevated PA pressure 45 mmHg
- Pulmonary valve is normal

Great Vessels: Aortic root 35 mm, ascending aorta 40 mm.

Moderate pulmonary hypertension present with interventricular septal flattening in systole.

IMPRESSION: Moderate LV hypertrophy with mild systolic dysfunction. Moderate pulmonary hypertension. Mild aortic stenosis, moderate mitral regurgitation.
            """
        },
        {
            "patient_id": "ECHO_003",
            "study_date": "2024-01-17", 
            "echo_report_text": """
PEDIATRIC ECHOCARDIOGRAM REPORT

Patient: 8-year-old male
Indication: Heart murmur

FINDINGS:
Left Ventricle: Normal left ventricular size and systolic function, EF 65%.

Right Ventricle: Mild right ventricular dilation with normal systolic function.

Atria: Normal atrial sizes.

Septal Defects: 
- Small perimembranous ventricular septal defect, 4 mm, with left-to-right shunt
- Peak gradient across VSD 65 mmHg
- No atrial septal defect

Valves: All valves are structurally normal and competent.

Great Vessels: Normal aortic arch, no coarctation. Patent ductus arteriosus is absent.

Mild elevation of right heart pressures secondary to VSD.

IMPRESSION: Small perimembranous VSD with left-to-right shunt. Mild RV dilation. Normal valves and great vessels.
            """
        }
    ]
    
    df = pd.DataFrame(sample_reports)
    sample_file = project_root / "data" / "sample_echo_reports_openai.csv"
    sample_file.parent.mkdir(exist_ok=True)
    df.to_csv(sample_file, index=False)
    
    print(f"✅ Created sample data: {sample_file}")
    print(f"📊 Sample data shape: {df.shape}")
    print(f"📋 Columns: {list(df.columns)}")
    
    return str(sample_file), df

# Create sample data
sample_file_path, sample_df = create_sample_echo_data()

# Display sample data
print(f"\n📊 Sample Data Preview:")
print(sample_df[["patient_id", "study_date"]].to_string(index=False))

# Cost estimation
def estimate_processing_cost(df, model_name, max_tokens):
    """Estimate OpenAI API costs for processing."""
    # Approximate pricing (as of 2024 - check current pricing)
    pricing = {
        "gpt-4o": {"input": 0.0050, "output": 0.0150},  # per 1K tokens
        "gpt-4o-mini": {"input": 0.00015, "output": 0.0006},
        "gpt-4": {"input": 0.0300, "output": 0.0600},
        "gpt-3.5-turbo": {"input": 0.0005, "output": 0.0015}
    }
    
    if model_name not in pricing:
        return "Pricing unknown"
    
    # Rough estimation
    avg_input_tokens = 1000  # Estimated tokens per echo report
    estimated_output_tokens = max_tokens * 0.7  # Assume 70% of max tokens used
    
    total_input_tokens = len(df) * avg_input_tokens
    total_output_tokens = len(df) * estimated_output_tokens
    
    input_cost = (total_input_tokens / 1000) * pricing[model_name]["input"]
    output_cost = (total_output_tokens / 1000) * pricing[model_name]["output"]
    total_cost = input_cost + output_cost
    
    return {
        "total_cost": total_cost,
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_input_tokens": total_input_tokens,
        "total_output_tokens": total_output_tokens
    }

# Show cost estimation
cost_estimate = estimate_processing_cost(sample_df, CONFIG["OPENAI_MODEL"], CONFIG["MAX_TOKENS"])
if isinstance(cost_estimate, dict):
    print(f"\n💰 Estimated Processing Cost ({CONFIG['OPENAI_MODEL']}):")
    print(f"   Total estimated cost: ${cost_estimate['total_cost']:.4f}")
    print(f"   Input tokens: ~{cost_estimate['total_input_tokens']:,}")
    print(f"   Output tokens: ~{int(cost_estimate['total_output_tokens']):,}")
    print(f"   ⚠️  This is a rough estimate - actual costs may vary")

# Option to use your own data (uncomment and modify the path)
# input_file_path = "/path/to/your/echo_reports.csv"
# input_df = pd.read_csv(input_file_path)

# Set the input file to use
input_file_path = sample_file_path
CONFIG["INPUT_FILE"] = input_file_path

## 5. Run OpenAI Extraction

Execute the cloud-based extraction using OpenAI's API. This will send data to OpenAI's servers for processing.

In [ ]:
async def run_openai_extraction():
    """Run extraction using OpenAI API."""
    if not api_available:
        print("❌ Cannot proceed - OpenAI API key not configured")
        return None, None
    
    print("🤖 Starting OpenAI Cloud Extraction")
    print("=" * 40)
    print(f"   Model: {CONFIG['OPENAI_MODEL']}")
    print(f"   Batch size: {CONFIG['BATCH_SIZE']}")
    print(f"   Temperature: {CONFIG['TEMPERATURE']}")
    print(f"   Records to process: {len(sample_df)}")
    print(f"   ⚠️  Data will be sent to OpenAI's servers")
    
    try:
        # Configure OpenAI extraction
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        output_file = str(CONFIG["OUTPUT_DIR"] / f"echo_openai_{timestamp}.xlsx")
        
        extraction_config = ProcessingConfig(
            input_file=CONFIG["INPUT_FILE"],
            text_column=CONFIG["TEXT_COLUMN"],
            uid_column=CONFIG["UID_COLUMN"],
            pydantic_model=EchoReport,
            generator_type="openai",
            model_name=CONFIG["OPENAI_MODEL"],
            experiment_label="openai_echo_extraction",
            batch_size=CONFIG["BATCH_SIZE"],
            backup_interval=CONFIG["BACKUP_INTERVAL"],
            max_retries=CONFIG["MAX_RETRIES"],
            temperature=CONFIG["TEMPERATURE"],
            max_tokens=CONFIG["MAX_TOKENS"],
            system_message=CONFIG["SYSTEM_MESSAGE"],
            pre_prompt=CONFIG["PRE_PROMPT"],
            output_file=output_file
        )
        
        print(f"\n🚀 Starting extraction process...")
        print(f"   This may take a few moments due to API rate limits...")
        
        # Run extraction
        processor = ExtraCTOpsProcessor(extraction_config)
        await processor.process_batch(open_file=False)
        
        # Calculate results
        successful = len([r for r in processor.results if r.success])
        failed = len([r for r in processor.results if not r.success])
        avg_time = sum(r.execution_time for r in processor.results) / len(processor.results) if processor.results else 0
        
        print(f"\n✅ OpenAI extraction completed!")
        print(f"   ✓ Successful: {successful}")
        print(f"   ✗ Failed: {failed}")
        print(f"   📊 Success rate: {(successful/len(processor.results)*100):.1f}%")
        print(f"   ⏱️ Average time: {avg_time:.2f}s per report")
        print(f"   📁 Output file: {output_file}")
        
        return processor, extraction_config
        
    except Exception as e:
        print(f"❌ Extraction failed: {e}")
        import traceback
        traceback.print_exc()
        return None, None

# Confirmation before running (to avoid accidental API charges)
print("🔍 READY TO RUN EXTRACTION")
print("=" * 30)
print("⚠️  This will:")
print("   • Send data to OpenAI's servers")
print("   • Incur API charges")
print("   • Process all echo reports in the dataset")

# Uncomment the next line to run the extraction
# processor, extraction_config = await run_openai_extraction()

# Comment out this line if you've uncommented the line above
print("\n⏸️ Extraction ready but not started")
print("   Uncomment the line above to run the actual extraction")
processor, extraction_config = None, None

## 6. Analyze Results

Examine the extraction results and display key statistics and sample outputs.

In [ ]:
def analyze_openai_results(processor, config):
    """Analyze and display extraction results."""
    if not processor or not processor.results:
        print("❌ No extraction results to analyze")
        return
    
    print("📊 OPENAI EXTRACTION ANALYSIS")
    print("=" * 40)
    
    # Overall statistics
    total_results = len(processor.results)
    successful = len([r for r in processor.results if r.success])
    failed = len([r for r in processor.results if not r.success])
    avg_time = sum(r.execution_time for r in processor.results) / total_results
    total_time = sum(r.execution_time for r in processor.results)
    
    print(f"📈 Overall Performance:")
    print(f"   Total reports processed: {total_results}")
    print(f"   Successful extractions: {successful}")
    print(f"   Failed extractions: {failed}")
    print(f"   Success rate: {(successful/total_results)*100:.1f}%")
    print(f"   Average processing time: {avg_time:.2f}s per report")
    print(f"   Total processing time: {total_time:.1f}s")
    
    # Quality analysis for successful extractions
    successful_results = [r for r in processor.results if r.success and r.extracted_data]
    if successful_results:
        print(f"\n🎯 Quality Analysis:")
        
        # Calculate average number of extracted fields
        field_counts = [len(r.extracted_data) for r in successful_results]
        avg_fields = sum(field_counts) / len(field_counts)
        max_fields = max(field_counts)
        min_fields = min(field_counts)
        
        print(f"   Average fields extracted: {avg_fields:.1f}")
        print(f"   Max fields extracted: {max_fields}")
        print(f"   Min fields extracted: {min_fields}")
        
        # Show sample successful extraction
        sample = successful_results[0]
        print(f"\n🔍 Sample Successful Extraction:")
        print(f"   Patient ID: {sample.uid}")
        print(f"   Processing time: {sample.execution_time:.2f}s")
        print(f"   Extracted fields: {len(sample.extracted_data)}")
        
        # Show some extracted fields
        if sample.extracted_data:
            print(f"   Sample extracted data:")
            field_count = 0
            for field_name, value in sample.extracted_data.items():
                if value and field_count < 5:  # Show first 5 non-empty fields
                    print(f"      {field_name}: {value}")
                    field_count += 1
            if len(sample.extracted_data) > 5:
                print(f"      ... and {len(sample.extracted_data) - 5} more fields")
    
    # Show failed extractions if any
    failed_results = [r for r in processor.results if not r.success]
    if failed_results:
        print(f"\n⚠️ Failed Extractions:")
        for failed in failed_results[:3]:  # Show first 3 failures
            print(f"   Patient ID: {failed.uid}")
            print(f"   Error: {failed.error}")
        if len(failed_results) > 3:
            print(f"   ... and {len(failed_results) - 3} more failures")
    
    # Load and preview output file
    if config and config.output_file:
        try:
            output_df = pd.read_excel(config.output_file)
            print(f"\n📁 Output File Analysis:")
            print(f"   File: {config.output_file}")
            print(f"   Shape: {output_df.shape}")
            print(f"   Columns: {len(output_df.columns)}")
            
            # Show extraction status distribution
            status_col = f"{config.experiment_label}_status"
            if status_col in output_df.columns:
                status_counts = output_df[status_col].value_counts()
                print(f"   Status distribution:")
                for status, count in status_counts.items():
                    print(f"      {status}: {count}")
                    
        except Exception as e:
            print(f"   ❌ Error loading output file: {e}")

# Analyze results if extraction was successful
if processor and extraction_config:
    analyze_openai_results(processor, extraction_config)
else:
    print("⏸️ No results to analyze - extraction was not run")
    print("   Run the extraction in the previous cell to see results here")

## 7. Save Summary Report

Create a comprehensive summary report of the extraction session.

In [ ]:
def save_openai_summary_report(processor, config):
    """Save a comprehensive summary report."""
    if not processor or not processor.results:
        print("❌ No results to save")
        return
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Calculate detailed statistics
    total_results = len(processor.results)
    successful = len([r for r in processor.results if r.success])
    failed = len([r for r in processor.results if not r.success])
    avg_time = sum(r.execution_time for r in processor.results) / total_results
    total_time = sum(r.execution_time for r in processor.results)
    
    # Calculate quality metrics for successful extractions
    successful_results = [r for r in processor.results if r.success and r.extracted_data]
    if successful_results:
        field_counts = [len(r.extracted_data) for r in successful_results]
        avg_fields = sum(field_counts) / len(field_counts)
        max_fields = max(field_counts)
        min_fields = min(field_counts)
    else:
        avg_fields = max_fields = min_fields = 0
    
    # Create comprehensive summary
    summary_report = {
        "extraction_session": {
            "timestamp": timestamp,
            "generator": "openai",
            "model": CONFIG["OPENAI_MODEL"],
            "cloud_processing": True,
            "input_file": CONFIG["INPUT_FILE"],
            "output_file": config.output_file if config else None,
            "processing_statistics": {
                "total_reports": total_results,
                "successful_extractions": successful,
                "failed_extractions": failed,
                "success_rate_percent": (successful / total_results) * 100,
                "average_time_per_report_seconds": avg_time,
                "total_processing_time_seconds": total_time
            },
            "quality_metrics": {
                "average_fields_extracted": avg_fields,
                "max_fields_extracted": max_fields,
                "min_fields_extracted": min_fields,
                "successful_extractions_with_data": len(successful_results)
            },
            "configuration": {
                "model": CONFIG["OPENAI_MODEL"],
                "batch_size": CONFIG["BATCH_SIZE"],
                "temperature": CONFIG["TEMPERATURE"],
                "max_tokens": CONFIG["MAX_TOKENS"],
                "backup_interval": CONFIG["BACKUP_INTERVAL"],
                "max_retries": CONFIG["MAX_RETRIES"]
            }
        },
        "pydantic_model": {
            "name": "EchoReport",
            "total_possible_fields": len(EchoReport.model_fields),
            "field_categories": list(EchoReport.model_fields.keys())[:10]  # Sample fields
        },
        "cost_and_usage": {
            "estimated_cost_note": "Check OpenAI usage dashboard for actual costs",
            "model_used": CONFIG["OPENAI_MODEL"],
            "processing_method": "Cloud API"
        }
    }
    
    # Save summary as JSON
    summary_file = CONFIG["OUTPUT_DIR"] / f"openai_extraction_summary_{timestamp}.json"
    with open(summary_file, 'w') as f:
        json.dump(summary_report, f, indent=2, default=str)
    
    print(f"📋 Summary Report Saved")
    print(f"   File: {summary_file}")
    print(f"   Success rate: {summary_report['extraction_session']['processing_statistics']['success_rate_percent']:.1f}%")
    print(f"   Total time: {summary_report['extraction_session']['processing_statistics']['total_processing_time_seconds']:.1f}s")
    print(f"   Avg fields extracted: {summary_report['extraction_session']['quality_metrics']['average_fields_extracted']:.1f}")
    
    return summary_report

# Save summary report if extraction was successful
if processor and extraction_config:
    summary_report = save_openai_summary_report(processor, extraction_config)
    
    print(f"\n🎉 OPENAI EXTRACTION COMPLETE!")
    print(f"📁 All files saved to: {CONFIG['OUTPUT_DIR']}")
    
    # List generated files
    output_files = list(CONFIG["OUTPUT_DIR"].glob("*"))
    print(f"📋 Generated files ({len(output_files)}):")
    for file_path in sorted(output_files):
        file_size = file_path.stat().st_size / 1024  # KB
        print(f"   📄 {file_path.name} ({file_size:.1f} KB)")
    
    print(f"\n💰 Don't forget to check your OpenAI usage dashboard for actual costs!")
    print(f"🔗 https://platform.openai.com/usage")
else:
    print("⏸️ No summary to save - extraction was not completed")
    print("   Run the extraction above to generate results and summary")

## 8. Model Comparison (Optional)

Quick utility to compare different OpenAI models on a single sample for quality assessment.

In [ ]:
async def compare_openai_models(sample_text: str, models: List[str] = None):
    """Compare different OpenAI models on a single sample text."""
    if not api_available:
        print("❌ Cannot compare models - API key not configured")
        return
    
    if models is None:
        models = ["gpt-4o-mini", "gpt-4o"]  # Default comparison
    
    print("🔍 MODEL COMPARISON")
    print("=" * 30)
    print(f"Testing models: {', '.join(models)}")
    print(f"Sample text length: {len(sample_text)} characters")
    
    results = {}
    
    for model in models:
        print(f"\n🤖 Testing {model}...")
        
        try:
            # Create a quick test config
            timestamp = datetime.now().strftime('%H%M%S')
            test_config = ProcessingConfig(
                input_file=CONFIG["INPUT_FILE"],
                text_column=CONFIG["TEXT_COLUMN"],
                uid_column=CONFIG["UID_COLUMN"],
                pydantic_model=EchoReport,
                generator_type="openai",
                model_name=model,
                experiment_label=f"test_{model.replace('-', '_')}",
                batch_size=1,
                backup_interval=1,
                max_retries=1,
                temperature=CONFIG["TEMPERATURE"],
                max_tokens=CONFIG["MAX_TOKENS"],
                system_message=CONFIG["SYSTEM_MESSAGE"],
                pre_prompt=CONFIG["PRE_PROMPT"],
                output_file=str(CONFIG["OUTPUT_DIR"] / f"test_{model}_{timestamp}.xlsx")
            )
            
            # Create temp CSV with just one sample
            temp_df = pd.DataFrame([{
                CONFIG["UID_COLUMN"]: "TEST_001",
                CONFIG["TEXT_COLUMN"]: sample_text
            }])
            temp_file = CONFIG["OUTPUT_DIR"] / f"temp_test_{timestamp}.csv"
            temp_df.to_csv(temp_file, index=False)
            test_config.input_file = str(temp_file)
            
            # Run extraction
            start_time = datetime.now()
            processor = ExtraCTOpsProcessor(test_config)
            await processor.process_batch(open_file=False)
            end_time = datetime.now()
            
            # Analyze result
            if processor.results and len(processor.results) > 0:
                result = processor.results[0]
                results[model] = {
                    "success": result.success,
                    "execution_time": result.execution_time,
                    "extracted_fields": len(result.extracted_data) if result.extracted_data else 0,
                    "error": result.error if not result.success else None
                }
                
                if result.success:
                    print(f"   ✅ Success!")
                    print(f"   ⏱️ Time: {result.execution_time:.2f}s")
                    print(f"   📊 Fields extracted: {len(result.extracted_data) if result.extracted_data else 0}")
                else:
                    print(f"   ❌ Failed: {result.error}")
            
            # Cleanup temp file
            temp_file.unlink(missing_ok=True)
            
        except Exception as e:
            print(f"   ❌ Error: {e}")
            results[model] = {"success": False, "error": str(e)}
    
    # Summary comparison
    print(f"\n📊 COMPARISON SUMMARY:")
    print("-" * 50)
    for model, result in results.items():
        if result["success"]:
            print(f"{model:15} | ✅ | {result['execution_time']:6.2f}s | {result['extracted_fields']:3d} fields")
        else:
            print(f"{model:15} | ❌ | Error: {result.get('error', 'Unknown')}")

# Example usage (uncomment to run model comparison)
if api_available and sample_df is not None:
    print("🔍 MODEL COMPARISON READY")
    print("Uncomment the lines below to compare models on a sample text:")
    print()
    
    # Get first sample text
    sample_text = sample_df.iloc[0][CONFIG["TEXT_COLUMN"]]
    
    # Uncomment to run comparison
    # await compare_openai_models(sample_text, ["gpt-4o-mini", "gpt-4o"])
    
    print("⏸️ Model comparison ready but not started")
else:
    print("⏸️ Model comparison not available - need API key and sample data")

## 🎯 Summary and Next Steps

### ✅ What This Notebook Accomplishes

1. **Cloud-Based Processing**: Leverages OpenAI's powerful models for high-quality extraction
2. **Cost Management**: Provides cost estimation and model selection options
3. **Quality Analysis**: Detailed analysis of extraction quality and performance
4. **Production Ready**: Robust error handling and comprehensive reporting
5. **Model Comparison**: Optional comparison between different OpenAI models

### 📊 Key Benefits of OpenAI Extraction

- **🎯 High Accuracy**: State-of-the-art models typically provide superior extraction quality
- **⚡ Scalability**: Cloud infrastructure handles large datasets efficiently
- **🔧 Model Variety**: Choose between different models based on quality/cost requirements
- **📈 Reliability**: Robust API with high uptime and performance
- **🚀 Latest Technology**: Access to cutting-edge language models

### 🚀 Next Steps

1. **Production Deployment**: Use this notebook as a template for production workflows
2. **Cost Optimization**: Monitor usage and optimize model selection for your use case
3. **Quality Validation**: Compare results with ground truth or expert annotations
4. **Integration**: Integrate extracted data into clinical databases or research workflows
5. **Comparison**: Run the Ollama notebook to compare cloud vs local processing

### 💡 Best Practices

- **Model Selection**: Use gpt-4o-mini for cost-effective processing, gpt-4o for maximum quality
- **Batch Size**: Higher batch sizes for efficiency, but respect rate limits
- **Error Handling**: Always implement retry logic for production use
- **Cost Monitoring**: Regular monitoring of OpenAI usage dashboard
- **Data Privacy**: Review OpenAI's data privacy policies for sensitive medical data

### 💰 Cost Management Tips

- **Start Small**: Test with small datasets first
- **Model Choice**: gpt-4o-mini is often sufficient for structured extraction
- **Token Optimization**: Optimize prompts to reduce token usage
- **Batch Processing**: Use appropriate batch sizes to minimize overhead
- **Monitoring**: Set up usage alerts in OpenAI dashboard

---

**Ready for high-quality cloud-based clinical data extraction! 🤖✨**